## Preprocessing
1) Loads both validation and test datasets
2) Checks the entity distribution 
3) Inserts emails into the datasets
4) Saves the new modified datasets in the "datasets" folder 

In [20]:
import re
import json
import random


DOMAINS = ['gmail.com', 'yahoo.com', 'outlook.com', 'live.com', 'hotmail.com']

SEPARATORS = ['-', '_', '.', '']


def load_data():
    with open('../datasets/data.json', 'r', encoding='utf-8') as file:
        train_data = json.load(file)
    
    with open('../datasets/test_data.json', 'r', encoding='utf-8') as file:
        test_data = json.load(file)

    print(f"Loaded training set with {len(train_data)} lines and testing set with {len(test_data)} lines.")
    return train_data, test_data

train_data, test_data = load_data()

def analyze_distribution(data, label):
    """ Checks the tag distributions in the dataset """
    counts = {}
    for entry in data:
        for tag in entry['ner_tags']:
            if tag not in counts:
                counts[tag] = 0
            counts[tag] += 1
    print()
    print(f"\033[1mTag Distribution for {label}\033[0m")
    total = sum(counts.values())

    for tag, count in counts.items():
        print(f"{tag} : {count} ({count / total * 100 :.2f}%)")

    if 'B-EMAIL' not in counts:
        print("No emails in dataset, augmentation required.")


def extract_names(data):
    """ Extracts names from the given dataset and returns two lists, first_names and last_names """

    first_names = set()
    last_names = set()
    ignore = {'is', 'poor', 'a', 'the', 'that'}

    for entry in data:
        tags = entry['ner_tags']
        tokens = entry['tokens']

        current_name = []
        for token, tag in zip(tokens, tags):
            token = token.lower()
            if tag == 'B-PER':
                current_name = [token]

            elif tag == 'I-PER':
                current_name.append(token)

            else:
                if current_name:                    # Condition to jump into if name has ended
                    if current_name[0] not in ignore:
                        first_names.add(current_name[0])
                        for name in current_name[1:]:
                            if name not in ignore:
                                last_names.add(name)
                current_name = []

        if current_name:                       # To resolve edge case where sequence ends with name 
            if current_name[0] not in ignore:
                first_names.add(current_name[0])
                for name in current_name[1:]:
                    if name not in ignore:
                        last_names.add(name)

    return list(first_names), list(last_names)                      

first_names, last_names = extract_names(train_data)

def clean_name(name):
    return re.sub(r'[^a-z]', '', name.lower())  # To remove special characters for model training 

def generate_email(first, last):
    """ Generates a realistic synthetic email address """
    first = clean_name(first)
    last = clean_name(last)

    if not first:
        first = random.choice(first_names)
    if not last:
        last = random.choice(last_names)
    sep = random.choice(SEPARATORS)
    domain = random.choice(DOMAINS)
    randnum = random.randint(1, 999)
    pattern = random.choice([
        f"{first}{sep}{last}",
        f"{first[0]}{sep}{last}",
        f"{first}{sep}{last[0]}",
        f"{first}",
        f"{first}{randnum}",
        f"{last}{randnum}",
        f"{first}{sep}{last}{randnum}",
        f"{first}{randnum}{last}",
        f"{last}{first}", 
        f"{last}{sep}{first}",
        f"{first[0]}{last}{randnum}",       
        f"{first}{sep}{last[0]}{randnum}"
    ])
 
    return f"{pattern}@{domain}"


def insert_email_in_entry(entry, insertion_prob=0.4):
    lang = entry['lang']
    tags = entry["ner_tags"]
    tokens = entry["tokens"]
    new_tokens = []
    new_tags = []
    current_name = []
    
    for token, tag in zip(tokens, tags):
        if tag == 'B-PER':
            if current_name:
                process_name_and_email(current_name, new_tokens, new_tags, insertion_prob)
            current_name = [token]

        elif tag == 'I-PER':
            current_name.append(token)
            
        else:
            if current_name:
                process_name_and_email(current_name, new_tokens, new_tags, insertion_prob)
                current_name = []
            
            # Append other tags and tokens as is
            new_tags.append(tag)
            new_tokens.append(token)

    # If sequence ends with a name
    if current_name:
        process_name_and_email(current_name, new_tokens, new_tags, insertion_prob)

    new_sequence = ' '.join(new_tokens)
    return {'lang': lang, 'ner_tags': new_tags, 'sequence': new_sequence, 'tokens': new_tokens}


def process_name_and_email(name, new_tokens, new_tags, insertion_prob):

    new_tokens.extend(name)
    new_tags.extend(['B-PER'] + ['I-PER'] * (len(name) - 1))

    if random.random() < insertion_prob:

        if random.random() < 0.5:   # Replacement prob, person's actual name will be used to generate an email half the time
            email = generate_email(name[0].lower(), name[-1].lower())

        else:
            email = generate_email(random.choice(first_names), random.choice(last_names))

        connectors = ['at', 'at', 'at', '(', '(', 'email:', ':', '<', '->', '']
        connector = random.choice(connectors)
        
        if connector != '':
            new_tags.append('O')
            new_tokens.append(connector)

        new_tags.append('B-EMAIL')
        new_tokens.append(email)

        if connector == '<':
            new_tags.append('O')
            new_tokens.append('>')

        elif connector == '(':
            new_tags.append('O')
            new_tokens.append(')')


def insert_emails(data):
    modified_data = []
    for entry in data:
        modified_data.append(insert_email_in_entry(entry))
    return modified_data

def validate_dataset(data, label):
    for i, entry in enumerate(data):
        if len(entry['tokens']) != len(entry['ner_tags']):
            raise ValueError(f"Length mismatch at entry {i}")
    print(f"Validation passed for {label}!")


def create_agumented_dataset():
    new_train_data = insert_emails(train_data)
    new_test_data = insert_emails(test_data)
    validate_dataset(new_train_data, "training set")
    validate_dataset(new_test_data, "testing set")
    with open('../datasets/train_data_modified.json', 'w', encoding='utf-8') as file:
        json.dump(new_train_data, file, indent=4)
    
    with open('../datasets/test_data_modified.json', 'w', encoding='utf-8') as file:
        json.dump(new_test_data, file, indent=4)

    return new_train_data, new_test_data


def main(): 
    new_train_data, new_test_data = create_agumented_dataset()
    analyze_distribution(new_train_data, "training set")
    analyze_distribution(new_test_data, "testing set")
    print("\n\033[1mAugmented Dataset Analysis for training set:\033[0m")
    for i in range(10):
        emails = []
        tokens = new_train_data[i]['tokens']
        tags = new_train_data[i]['ner_tags']

        for j, tag in enumerate(tags):
            if tag == 'B-EMAIL':
                emails.append(tokens[j])

        print(f"\nEmail(s) added: {emails}")
        print(f"New Sequence: {new_train_data[i]['sequence']}")
    
    print("\n\033[1mAugmented Dataset Analysis for testing set:\033[0m")
    for i in range(10):
        emails = []
        tokens = new_test_data[i]['tokens']
        tags = new_test_data[i]['ner_tags']

        for j, tag in enumerate(tags):
            if tag == 'B-EMAIL':
                emails.append(tokens[j])
                
        print(f"\nEmail(s) added: {emails}")
        print(f"New Sequence: {new_test_data[i]['sequence']}")

main()
        





Loaded training set with 28516 lines and testing set with 3650 lines.
Validation passed for training set!
Validation passed for testing set!

Tag Distribution for training set
O : 658247 (88.48%)
B-PER : 40270 (5.41%)
I-PER : 29460 (3.96%)
B-EMAIL : 15952 (2.14%)

Tag Distribution for testing set
O : 80427 (87.66%)
B-PER : 5210 (5.68%)
I-PER : 3956 (4.31%)
B-EMAIL : 2157 (2.35%)

Augmented Dataset Analysis for training set:

Email(s) added: ['curleyj703@live.com']
New Sequence: Since then , only Terry Bradshaw in 147 games , Joe Montana in 139 games , and Tom Brady at curleyj703@live.com in 131 games have reached 100 wins more quickly .

Email(s) added: []
New Sequence: He was portrayed by Anthony Perkins in the 1960 version of " Psycho " directed by Alfred Hitchcock and the " Psycho " franchise .

Email(s) added: []
New Sequence: The egg eventually hatches , revealing a baby Sharptooth .

Email(s) added: []
New Sequence: In the video Kelis is walking down a street in a large space sui